# Projeto — Análise Exploratória de Dados de Vendas

**Período analisado:** 1º de janeiro a 31 de dezembro de 2023  
**Fonte:** `vendas_supermercado_com_cliente.csv`  
As análises usam operações vistas nas aulas 1 a 5: leitura e seleção com Pandas, filtros, colunas, `groupby`, `agg` e gráficos com Seaborn/Matplotlib. As conclusões calculadas aparecem junto aos resultados.

Este notebook responde às dez perguntas do projeto. Cada seção apresenta o critério adotado e uma tabela ou gráfico produzido a partir dos dados. Valores monetários são apresentados na moeda/unidade registrada no arquivo, que não especifica moeda.

> **Limitação da base:** cada linha representa um produto em uma data, sem código de transação, cliente ou loja. Assim, “ticket médio” é tratado como valor médio de venda por registro (linha), não como valor médio de uma cesta. As associações observadas não demonstram causalidade.


In [ ]:
from pathlib import Path
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
from IPython.display import display

ARQUIVO = Path("vendas_supermercado_com_cliente.csv")
df = pd.read_csv(ARQUIVO, encoding="utf-8-sig")
df = df.sort_values("Data").reset_index(drop=True)
df["Mes"] = df["Data"].apply(lambda data: int(data[5:7]))
meses = {1:"Jan",2:"Fev",3:"Mar",4:"Abr",5:"Mai",6:"Jun",7:"Jul",8:"Ago",9:"Set",10:"Out",11:"Nov",12:"Dez"}
df["Mes_nome"] = df["Mes"].apply(lambda mes: meses[mes])
df["Dia_semana"] = df["Data"].apply(lambda data: datetime.strptime(data, "%Y-%m-%d").weekday())
nomes_dias = {0:"Segunda",1:"Terça",2:"Quarta",3:"Quinta",4:"Sexta",5:"Sábado",6:"Domingo"}
df["Dia_semana_nome"] = df["Dia_semana"].apply(lambda dia: nomes_dias[dia])
df["Promocao"] = df["Desconto"].gt(0)
print(f"Registros: {len(df):,} | Produtos: {df['Produto'].nunique()} | Categorias: {df['Categoria'].nunique()}")
print(f"Período: {df['Data'].min()} a {df['Data'].max()} | Valores ausentes: {int(df.isna().sum().sum())}")
display(df.head())
df.head()

## 1. Categorias mais vendidas e meses de pico

“Mais vendidas” é medido pela quantidade total de unidades. O pico mensal de cada categoria também usa unidades.

In [ ]:
cat_total = df.groupby("Categoria")["Quantidade"].sum().sort_values(ascending=False)
cat_mes = df.groupby(["Categoria", "Mes_nome"], as_index=False)["Quantidade"].sum()
mes_ordem = ["Jan","Fev","Mar","Abr","Mai","Jun","Jul","Ago","Set","Out","Nov","Dez"]
cat_mes["Mes_nome"] = pd.Categorical(cat_mes["Mes_nome"], categories=mes_ordem, ordered=True)
picos = cat_mes.sort_values("Quantidade", ascending=False).groupby("Categoria").head(1)
res = picos.merge(cat_total.rename("Unidades_ano"), on="Categoria").sort_values("Unidades_ano", ascending=False)
display(res.rename(columns={"Quantidade":"Unidades_no_pico","Mes_nome":"Mes_pico"}).reset_index(drop=True))
print("Produtos com maior quantidade anual:")
top_prod = df.groupby("Produto")["Quantidade"].sum().sort_values(ascending=False).head(10)
display(top_prod.rename("Unidades").to_frame())
fig,ax=plt.subplots(figsize=(9,5)); sns.barplot(x=cat_total.head(10).values,y=cat_total.head(10).index,ax=ax,color="#3B82A0")
ax.set(title="Categorias líderes em unidades vendidas",xlabel="Unidades no ano",ylabel=""); plt.tight_layout(); plt.show()
print(f"Conclusão: {cat_total.index[0]} liderou com {cat_total.iloc[0]:,} unidades; o mês de pico varia por categoria conforme a tabela acima.")

## 2. Média de vendas diárias por categoria

Somam-se os valores de venda por categoria em cada data e calcula-se a média diária ao longo do período.

In [ ]:
por_dia_cat = df.groupby(["Data", "Categoria"]).agg(Venda_do_dia=("Total_Venda","sum")).reset_index()
media_diaria_cat = por_dia_cat.groupby("Categoria")["Venda_do_dia"].mean().sort_values(ascending=False)
display(media_diaria_cat.rename("Venda_media_diaria").to_frame())
fig,ax=plt.subplots(figsize=(9,5)); sns.barplot(x=media_diaria_cat.values,y=media_diaria_cat.index,ax=ax,color="#4E79A7")
ax.set(title="Venda média diária por categoria",xlabel="Valor médio por dia",ylabel=""); plt.tight_layout(); plt.show()
print(f"Conclusão: {media_diaria_cat.idxmax()} teve a maior venda média diária ({media_diaria_cat.max():,.2f}).")


## 3. Sazonalidade dos produtos

A sazonalidade é estimada pelo coeficiente de variação (CV = desvio-padrão/média) das quantidades mensais de cada produto. CV alto indica maior variação relativa entre meses; com apenas um ano, não comprova um padrão sazonal recorrente.

In [ ]:
prod_mes = df.groupby(["Produto", "Mes"]).agg(Quantidade_mensal=("Quantidade","sum")).reset_index()
sazonal = prod_mes.groupby("Produto")["Quantidade_mensal"].agg(["mean","std"])
sazonal["CV_mensal"] = sazonal["std"] / sazonal["mean"]
extremos = prod_mes.sort_values("Quantidade_mensal").groupby("Produto").agg(Menor_mes=("Mes","first"),Maior_mes=("Mes","last")).reset_index()
sazonal = pd.merge(sazonal.reset_index(),extremos,on="Produto")
sazonal["Menor_mes"] = sazonal["Menor_mes"].apply(lambda mes: meses[mes])
sazonal["Maior_mes"] = sazonal["Maior_mes"].apply(lambda mes: meses[mes])
display(sazonal.sort_values("CV_mensal", ascending=False).head(15))
top_cv=sazonal.sort_values("CV_mensal",ascending=False).head(10).sort_values("CV_mensal").set_index("Produto")
fig,ax=plt.subplots(figsize=(9,5)); sns.barplot(x=top_cv.CV_mensal.values,y=top_cv.index,ax=ax,color="#F28E2B")
ax.set(title="Produtos com maior variação relativa mensal",xlabel="Coeficiente de variação mensal",ylabel=""); plt.tight_layout(); plt.show()
print(f"Conclusão: {sazonal.sort_values('CV_mensal',ascending=False).iloc[0]['Produto']} teve o maior CV mensal ({sazonal['CV_mensal'].max():.1%}). Com apenas um ano, isso mede variação dentro do ano, não recorrência sazonal.")

## 4. Faixas etárias com maior contribuição às vendas

Faixas conforme o enunciado: jovem (18–35), adulto (36–59) e idoso (60 anos ou mais). A faixa idoso inclui 60 anos para não deixar essa idade sem classificação.

In [ ]:
def classificar_idade(idade):
    if idade <= 35:
        return "Jovem (18–35)"
    elif idade <= 59:
        return "Adulto (36–59)"
    else:
        return "Idoso (60+)"
df["Faixa_etaria"] = df["Idade"].apply(classificar_idade)
faixa_vendas = df.groupby("Faixa_etaria").agg(Venda_total=("Total_Venda","sum"), Registros=("Total_Venda","count"), Idade_media=("Idade","mean"))
faixa_vendas["Participacao"] = faixa_vendas["Venda_total"] / df["Total_Venda"].sum()
display(faixa_vendas)
fig,ax=plt.subplots(figsize=(7,4)); sns.barplot(x=faixa_vendas.index.astype(str),y=faixa_vendas.Venda_total.values,ax=ax)
ax.set(title="Vendas totais por faixa etária",ylabel="Valor de venda",xlabel=""); plt.xticks(rotation=0); plt.tight_layout(); plt.show()
lider_faixa=faixa_vendas.Venda_total.idxmax(); print(f"Conclusão: {lider_faixa} contribuiu mais para a receita ({faixa_vendas.loc[lider_faixa,'Participacao']:.1%} do total).")

## 5. Relação entre idade e categoria mais vendida

Para cada faixa, a categoria líder é definida pela soma de unidades; a tabela também mostra sua participação nas unidades da faixa.

In [ ]:
faixa_cat = df.groupby(["Faixa_etaria","Categoria"]).agg(Unidades=("Quantidade","sum"), Venda=("Total_Venda","sum")).reset_index()
totais_faixa = faixa_cat.groupby("Faixa_etaria").agg(Unidades_faixa=("Unidades","sum")).reset_index()
faixa_cat = pd.merge(faixa_cat,totais_faixa,on="Faixa_etaria")
faixa_cat["Participacao_unidades_faixa"] = faixa_cat["Unidades"] / faixa_cat["Unidades_faixa"]
lideres = faixa_cat.loc[faixa_cat.groupby("Faixa_etaria")["Unidades"].idxmax()].sort_values("Faixa_etaria")
display(lideres)
fig,ax=plt.subplots(figsize=(11,5)); sns.barplot(data=faixa_cat,x="Categoria",y="Unidades",hue="Faixa_etaria",ax=ax)
ax.set(title="Unidades por categoria e faixa etária",ylabel="Unidades",xlabel=""); plt.xticks(rotation=35,ha="right"); plt.tight_layout(); plt.show()
lider_jovem=lideres[lideres["Faixa_etaria"]=="Jovem (18–35)"].iloc[0]["Categoria"]
lider_adulto=lideres[lideres["Faixa_etaria"]=="Adulto (36–59)"].iloc[0]["Categoria"]
lider_idoso=lideres[lideres["Faixa_etaria"]=="Idoso (60+)"].iloc[0]["Categoria"]
print(f"Conclusão — categoria líder por faixa: Jovem: {lider_jovem}; Adulto: {lider_adulto}; Idoso: {lider_idoso}.")

## 6. Distribuição das vendas por dia da semana

Compara-se a média de venda diária total por dia da semana, evitando que meses com mais ocorrências de um dia distorçam o total.

In [ ]:
dia = df.groupby(["Data","Dia_semana_nome"])["Total_Venda"].sum().reset_index()
ordem_dias = ["Segunda","Terça","Quarta","Quinta","Sexta","Sábado","Domingo"]
dist_dia = dia.groupby("Dia_semana_nome")["Total_Venda"].agg(Media_diaria="mean", Total="sum", Dias="count").reindex(ordem_dias)
display(dist_dia)
fig,ax=plt.subplots(figsize=(9,4)); sns.barplot(x=dist_dia.index,y=dist_dia.Media_diaria.values,ax=ax,color="#76B7B2")
ax.set(title="Venda média diária por dia da semana",ylabel="Valor médio por dia",xlabel=""); plt.xticks(rotation=0); plt.tight_layout(); plt.show()
mais_forte=dist_dia.Media_diaria.idxmax(); mais_fraco=dist_dia.Media_diaria.idxmin()
print(f"Conclusão: {mais_forte} teve a maior média diária ({dist_dia.loc[mais_forte,'Media_diaria']:,.2f}) e {mais_fraco} a menor ({dist_dia.loc[mais_fraco,'Media_diaria']:,.2f}).")


## 7. Vendas mensais antes e durante a Black Friday

Como a Black Friday de 2023 ocorreu em 24 de novembro, “durante” é representado por novembro completo; janeiro a outubro representam o período anterior. A média diária ajuda a comparar meses com quantidades diferentes de dias.

In [ ]:
mensal = df.groupby(["Mes","Mes_nome"]).agg(Venda_total=("Total_Venda","sum"), Unidades=("Quantidade","sum"), Registros=("Total_Venda","size")).reset_index()
mensal["Venda_media_diaria"] = mensal["Venda_total"] / mensal["Data"].nunique() if "Data" in mensal else mensal["Venda_total"] / df.groupby("Mes")["Data"].nunique().values
def classificar_periodo(mes):
    if mes < 11:
        return "Antes (jan–out)"
    elif mes == 11:
        return "Mês da Black Friday (nov)"
    else:
        return "Após (dez)"
mensal["Periodo"] = mensal["Mes"].apply(classificar_periodo)
display(mensal[["Mes_nome","Venda_total","Venda_media_diaria","Unidades","Periodo"]])
antes = mensal.loc[mensal.Mes<11,"Venda_media_diaria"].mean(); novembro = mensal.loc[mensal.Mes==11,"Venda_media_diaria"].iloc[0]
print(f"Média diária jan–out: {antes:,.2f} | novembro: {novembro:,.2f} | variação: {(novembro/antes-1):+.1%}")
fig,ax=plt.subplots(figsize=(10,4)); sns.lineplot(data=mensal,x="Mes",y="Venda_media_diaria",marker="o",ax=ax,color="#4E79A7")
ax.axvline(11,color="#E15759",linestyle="--",label="Black Friday em novembro"); ax.set_xticks(range(1,13),mes_ordem)
ax.set_ylabel("Venda média diária"); ax.set_title("Vendas mensais: média diária"); ax.legend(); ax.grid(alpha=.2); plt.tight_layout(); plt.show()
print(f"Conclusão: novembro ficou {abs(novembro/antes-1):.1%} {'acima' if novembro>antes else 'abaixo'} da média diária de janeiro a outubro.")

## 8. Ticket médio por faixa etária e categoria

Sem identificador de compra, usamos a média de `Total_Venda` por linha como **valor médio por registro de produto**. Essa medida não deve ser interpretada como ticket por cliente ou cesta.

In [ ]:
ticket = df.groupby(["Faixa_etaria","Categoria"]).agg(Ticket_medio_registro=("Total_Venda","mean")).reset_index()
display(ticket)
print("Média geral por faixa etária (valor por registro):")
display(df.groupby("Faixa_etaria").agg(Valor_medio_por_registro=("Total_Venda","mean")))
fig,ax=plt.subplots(figsize=(11,5)); sns.barplot(data=ticket,x="Categoria",y="Ticket_medio_registro",hue="Faixa_etaria",ax=ax)
ax.set_title("Valor médio por registro: faixa etária × categoria"); ax.set_xlabel(""); ax.set_ylabel("Valor médio"); plt.xticks(rotation=35,ha="right"); plt.tight_layout(); plt.show()
maior_ticket=ticket.sort_values("Ticket_medio_registro",ascending=False).iloc[0]
print(f"Conclusão: maior valor médio por registro em {maior_ticket['Faixa_etaria']} / {maior_ticket['Categoria']} ({maior_ticket['Ticket_medio_registro']:,.2f}).")

## 9. Semana da Black Friday versus semana do Natal

Compara-se a semana calendário de segunda a domingo que contém cada data: 20–26/11/2023 e 25–31/12/2023. Indicador principal: venda média diária total da loja.

In [ ]:
bf_semana = df[df["Data"].between("2023-11-20","2023-11-26")]
natal_semana = df[df["Data"].between("2023-12-25","2023-12-31")]
bf_por_dia = bf_semana.groupby("Data").agg(Venda_diaria=("Total_Venda","sum"))
natal_por_dia = natal_semana.groupby("Data").agg(Venda_diaria=("Total_Venda","sum"))
comp = pd.DataFrame({"Venda_total":[bf_por_dia["Venda_diaria"].sum(),natal_por_dia["Venda_diaria"].sum()],
                     "Media_diaria":[bf_por_dia["Venda_diaria"].mean(),natal_por_dia["Venda_diaria"].mean()]},
                    index=["Black Friday","Natal"])
display(comp)
vencedora=comp.Media_diaria.idxmax(); print(f"Maior média diária: {vencedora} ({comp.loc[vencedora,'Media_diaria']:,.2f}).")
fig,ax=plt.subplots(figsize=(7,4)); sns.barplot(x=comp.index,y=comp.Media_diaria.values,ax=ax)
ax.set(title="Média diária nas semanas da Black Friday e do Natal",ylabel="Valor médio por dia",xlabel=""); plt.xticks(rotation=0); plt.tight_layout(); plt.show()
print(f"Conclusão: a semana de {vencedora} superou a outra em {abs(comp.Media_diaria.iloc[0]-comp.Media_diaria.iloc[1]):,.2f} por dia, em média.")

## 10. Impacto observado dos descontos e produtos com maior aumento

Promoção é qualquer registro com desconto maior que zero. O efeito é descritivo: compara registros promocionais e sem promoção. O aumento por produto usa quantidade média diária nos dias em que houve desconto no produto versus os demais dias; exige ao menos um dia em cada grupo.

In [ ]:
impacto = df.groupby("Promocao").agg(Registros=("Total_Venda","size"), Desconto_medio=("Desconto","mean"), Quantidade_media=("Quantidade","mean"), Venda_media=("Total_Venda","mean"), Venda_soma=("Total_Venda","sum"))
impacto = impacto.reset_index()
impacto["Situacao"] = impacto["Promocao"].apply(lambda promocao: "Com desconto" if promocao else "Sem desconto")
impacto = impacto.set_index("Situacao")
display(impacto)
com_promocao = df[df["Promocao"]].groupby("Produto").agg(Qtd_com_promocao=("Quantidade","mean")).reset_index()
sem_promocao = df[~df["Promocao"]].groupby("Produto").agg(Qtd_sem_promocao=("Quantidade","mean")).reset_index()
por_prod = pd.merge(com_promocao,sem_promocao,on="Produto")
por_prod["Aumento_qtd_media_%"] = (por_prod["Qtd_com_promocao"] / por_prod["Qtd_sem_promocao"] - 1) * 100
por_prod = por_prod.set_index("Produto")
display(por_prod.sort_values("Aumento_qtd_media_%",ascending=False).head(15))
print("Interpretação: diferença observacional; não é possível afirmar causalidade.")
fig,axes=plt.subplots(1,2,figsize=(10,4))
sns.barplot(x=impacto.index,y=impacto["Quantidade_media"].values,ax=axes[0]); axes[0].set_title("Quantidade média por registro")
sns.barplot(x=impacto.index,y=impacto["Venda_media"].values,ax=axes[1]); axes[1].set_title("Venda média por registro")
for ax in axes: ax.set_xlabel(""); ax.tick_params(axis="x",rotation=20)
plt.tight_layout(); plt.show()
melhor=por_prod.sort_values("Aumento_qtd_media_%",ascending=False).head(10).sort_values("Aumento_qtd_media_%")
fig,ax=plt.subplots(figsize=(9,5)); sns.barplot(x=melhor["Aumento_qtd_media_%"].values,y=melhor.index,ax=ax,color="#59A14F")
ax.set(title="Produtos com maior variação de quantidade em dias promocionais",xlabel="Variação vs. dias sem desconto (%)",ylabel=""); plt.tight_layout(); plt.show()
delta=impacto.loc["Com desconto","Quantidade_media"]-impacto.loc["Sem desconto","Quantidade_media"]
print(f"Conclusão: a quantidade média por registro foi {abs(delta):.2f} unidades {'maior' if delta>0 else 'menor' if delta<0 else 'igual'} em registros promocionais. {por_prod['Aumento_qtd_media_%'].idxmax()} teve a maior alta estimada ({por_prod['Aumento_qtd_media_%'].max():+.1f}%). Associação descritiva, não causal.")

## Conclusão

As tabelas e gráficos acima mostram os resultados por categoria, produto, período e faixa etária. Ao interpretar os resultados, considere que a base contém registros agregados por produto e data, cobre somente um ano e não informa moeda nem transações/cliente identificável. Portanto, padrões temporais e relações com descontos devem ser tratados como evidência descritiva para orientar investigações, não como estimativas causais.